In [12]:
import pandas as pd
df = pd.read_csv('gravityperformance.csv')
links = df['url'].to_list()

with open('urls.txt', 'r') as f:
    urls = f.read().splitlines()

temp = set(urls) - set(links)
with open('links.txt', 'w') as f:
    for link in temp:
        f.write(f'{link}\n')
print('length of temp', len(temp))

length of temp 0


In [6]:
import pandas as pd
import uuid

# -----------------------------
# CONFIG
# -----------------------------
INPUT_CSV = "gravityperformance2.csv"
OUTPUT_CSV = "gravityperformance_full_data2.csv"

DEFAULT_TYPE = "GOODS"
DEFAULT_COST = "0.00"
DEFAULT_WEIGHT = "0.01"

rows = []

# -----------------------------
# READ CSV (SAFE MODE)
# -----------------------------
df = pd.read_csv(INPUT_CSV, encoding="utf-8", skipinitialspace=True)

# Normalize column names
df.columns = df.columns.str.strip().str.lower()

for _, row in df.iterrows():
    try:
        title = str(row["title"]).strip()
        category = str(row["category"]).strip()
        description = str(row["description"]).strip()
        images_raw = str(row["images"]).strip()

        # -------- PRICE FIX --------
        price_raw = str(row["price"]).replace(",", "").strip()
        price = f"{float(price_raw):.2f}"

    except Exception as e:
        print("⚠️ Skipping row due to error:", e)
        continue

    product_id = str(uuid.uuid4())[:8]

    image_list = [img.strip() for img in images_raw.split(",") if img.strip()]

    # -----------------------------
    # MAIN PRODUCT ROW
    # -----------------------------
    rows.append({
        "id": product_id,
        "name": title,
        "type": DEFAULT_TYPE,
        "categ_id": category,
        "default_code": "",
        "barcode": "",
        "list_price": price,
        "standard_price": DEFAULT_COST,
        "weight": DEFAULT_WEIGHT,
        "description": description,
        "image_1920": image_list[0] if image_list else "",
        "Extra Product Media/Image": image_list[0] if image_list else "",
        "Extra Product Media/Name": "Main Image"
    })

    # -----------------------------
    # EXTRA IMAGES
    # -----------------------------
    for i, img in enumerate(image_list[1:3], start=1):
        rows.append({
            "id": product_id,
            "name": "",
            "type": "",
            "categ_id": "",
            "default_code": "",
            "barcode": "",
            "list_price": "",
            "standard_price": "",
            "weight": "",
            "description": "",
            "image_1920": "",
            "Extra Product Media/Image": img,
            "Extra Product Media/Name": f"Gallery Image {i}"
        })

# -----------------------------
# SAVE OUTPUT CSV
# -----------------------------
output_df = pd.DataFrame(rows)
output_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print("✅ CSV created successfully:", OUTPUT_CSV)


✅ CSV created successfully: gravityperformance_full_data2.csv


In [5]:
import pandas as pd
import uuid
import os
import re

# -----------------------------
# CONFIG
# -----------------------------
INPUT_CSV = "gravityperformance2.csv"
OUTPUT_DIR = "gravityperformance_full_data2"

DEFAULT_TYPE = "GOODS"
DEFAULT_COST = "0.00"
DEFAULT_WEIGHT = "0.01"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# -----------------------------
# READ CSV SAFELY
# -----------------------------
df = pd.read_csv(INPUT_CSV, encoding="utf-8", skipinitialspace=True)
df.columns = df.columns.str.strip().str.lower()

# Group rows by category
grouped = df.groupby("category")

for category, group in grouped:
    rows = []

    # Clean category name for filename
    safe_category = re.sub(r"[^a-zA-Z0-9_]+", "_", category.strip().lower())
    output_file = os.path.join(OUTPUT_DIR, f"{safe_category}.csv")

    for _, row in group.iterrows():
        try:
            title = str(row["title"]).strip()
            description = str(row["description"]).strip()
            images_raw = str(row["images"]).strip()

            price_raw = str(row["price"]).replace(",", "").strip()
            price = f"{float(price_raw):.2f}"

        except Exception as e:
            print(f"⚠️ Skipping row in '{category}' due to error:", e)
            continue

        product_id = str(uuid.uuid4())[:8]
        image_list = [img.strip() for img in images_raw.split(",") if img.strip()]

        # -----------------------------
        # MAIN PRODUCT ROW
        # -----------------------------
        rows.append({
            "id": product_id,
            "name": title,
            "type": DEFAULT_TYPE,
            "categ_id": category,
            "default_code": "",
            "barcode": "",
            "list_price": price,
            "standard_price": DEFAULT_COST,
            "weight": DEFAULT_WEIGHT,
            "description": description,
            "image_1920": image_list[0] if image_list else "",
            "Extra Product Media/Image": image_list[0] if image_list else "",
            "Extra Product Media/Name": "Main Image"
        })

        # -----------------------------
        # EXTRA IMAGES
        # -----------------------------
        for i, img in enumerate(image_list[1:3], start=1):
            rows.append({
                "id": product_id,
                "name": "",
                "type": "",
                "categ_id": "",
                "default_code": "",
                "barcode": "",
                "list_price": "",
                "standard_price": "",
                "weight": "",
                "description": "",
                "image_1920": "",
                "Extra Product Media/Image": img,
                "Extra Product Media/Name": f"Gallery Image {i}"
            })

    # -----------------------------
    # SAVE CATEGORY FILE
    # -----------------------------
    if rows:
        pd.DataFrame(rows).to_csv(
            output_file,
            index=False,
            encoding="utf-8-sig"
        )
        print(f"✅ Created: {output_file}")

print("🎉 All category files created successfully!")


✅ Created: gravityperformance_full_data2/lowering_springs.csv
🎉 All category files created successfully!


In [1]:
# The range goes to 7386 because the last number is not included
with open('image_links.txt', 'w') as f:
    for i in range(1, 7386):
        link = f"https://media.githubusercontent.com/media/ahmedkhaled115/image-storage/main/{i}.jpg"
        f.write(link + "\n")

print("Done! Check 'image_links.txt' in your folder.")

Done! Check 'image_links.txt' in your folder.


In [2]:
import pandas as pd
import numpy as np
import os

# 1. Setup Folder
output_folder = 'data'
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# 2. Load File
input_file = 'gravityperformance_full_data2.csv'
df = pd.read_csv(input_file, encoding='utf-8-sig')

# 3. Build Groups and Image URLs BEFORE clearing IDs
# Each time 'name' is not empty, it's a new product
df['product_group'] = df['name'].notnull().cumsum()

# Generate the GitHub image path using the existing ID
df['image'] = 'https://media.githubusercontent.com/media/ahmedkhaled115/image-storage/main/' + \
              df['id'].astype(str).str.split('.').str[0] + '.jpg'

# 4. Update Categories (Mapping based on Keywords)
def map_category(name):
    if pd.isna(name): return ""
    name = str(name).lower()
    
    if "radiator" in name: return "Radiators"
    if "downpipe" in name or "decat" in name or "fooler" in name or "spacer" in name: return "Downpipes"
    if "intercooler" in name: return "Intercoolers"
    if "induction kit" in name or "air kit" in name: return "Air Kits"
    if "spoiler" in name or "wing" in name: return "Spoilers And Wings"
    if "grille" in name: return "Grilles"
    if "spring" in name: return "Lowering Springs"
    if "arm" in name or "rod" in name or "camber" in name: return "Adjustable arms"
    if "bilstein" in name: return "Bilstein Suspension Coilovers"
    if "brace" in name: return "Strut Braces"
    if "diffuser" in name or "canard" in name or "spat" in name: return "Rear Diffusers"
    if "intake" in name or "induction" in name or "filter" in name: return "Air Induction"
    if "exhaust" in name or "catback" in name or "back box" in name: return "Exhaust"
    return "Exhaust" # Default

# Apply category mapping to main rows and fill down to gallery rows
df.loc[df['name'].notnull(), 'categ_id'] = df.loc[df['name'].notnull(), 'name'].apply(map_category)
df['categ_id'] = df.groupby('product_group')['categ_id'].transform('first')

# 5. Final Processing (Renaming & Cleaning)

# Rename "Gallery Image X" to "Name X"
df['Extra Product Media/Name'] = df['Extra Product Media/Name'].str.replace('Gallery Image ', 'Name ', regex=False)

# image_1920 is the first image of the group
df['image_1920'] = df.groupby('product_group')['image'].transform('first')

# Identify rows that are NOT the first row of a product
is_gallery_row = df.duplicated(subset=['product_group'])

# Clear metadata for gallery rows + Clear ID for ALL rows
cols_to_clear_gallery = ['name', 'categ_id', 'default_code', 'barcode', 'list_price', 
                         'standard_price', 'weight', 'description', 'image_1920']

df.loc[is_gallery_row, cols_to_clear_gallery] = ""
df['id'] = "" # Remove all values in column ID as requested

# Reorder columns
column_order = [
    'id', 'name', 'categ_id', 'default_code', 'barcode', 'list_price', 
    'standard_price', 'weight', 'description', 'image_1920', 'image', 'Extra Product Media/Name'
]
df_final = df[column_order]

# 6. Split into Files (30 products per file)
products_per_file = 30
unique_product_ids = df['product_group'].unique()

for i in range(0, len(unique_product_ids), products_per_file):
    batch_ids = unique_product_ids[i : i + products_per_file]
    file_subset = df_final[df['product_group'].isin(batch_ids)]
    
    file_num = (i // products_per_file) + 1
    file_path = os.path.join(output_folder, f'Products_Batch_{file_num}.xlsx')
    
    file_subset.to_excel(file_path, index=False)
    print(f"File {file_num} saved to /data/ folder.")

print("\nDone! All files created with updated categories and removed IDs.")

/var/folders/vp/xy1fyb4j5f594ppbf387zs8c0000gn/T/ipykernel_42406/445117915.py:61: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[is_gallery_row, cols_to_clear_gallery] = ""
/var/folders/vp/xy1fyb4j5f594ppbf387zs8c0000gn/T/ipykernel_42406/445117915.py:61: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[is_gallery_row, cols_to_clear_gallery] = ""
/var/folders/vp/xy1fyb4j5f594ppbf387zs8c0000gn/T/ipykernel_42406/445117915.py:61: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible d

File 1 saved to /data/ folder.
File 2 saved to /data/ folder.
File 3 saved to /data/ folder.
File 4 saved to /data/ folder.
File 5 saved to /data/ folder.
File 6 saved to /data/ folder.
File 7 saved to /data/ folder.
File 8 saved to /data/ folder.
File 9 saved to /data/ folder.
File 10 saved to /data/ folder.
File 11 saved to /data/ folder.
File 12 saved to /data/ folder.
File 13 saved to /data/ folder.
File 14 saved to /data/ folder.
File 15 saved to /data/ folder.
File 16 saved to /data/ folder.
File 17 saved to /data/ folder.
File 18 saved to /data/ folder.
File 19 saved to /data/ folder.
File 20 saved to /data/ folder.
File 21 saved to /data/ folder.
File 22 saved to /data/ folder.
File 23 saved to /data/ folder.
File 24 saved to /data/ folder.
File 25 saved to /data/ folder.
File 26 saved to /data/ folder.
File 27 saved to /data/ folder.
File 28 saved to /data/ folder.
File 29 saved to /data/ folder.
File 30 saved to /data/ folder.
File 31 saved to /data/ folder.
File 32 saved to 

In [7]:
import pandas as pd
import numpy as np
import os

# 1. Setup Folder
output_folder = 'data'
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# 2. Load File
input_file = 'gravityperformance_full_data2.csv'
df = pd.read_csv(input_file, encoding='utf-8-sig')

# 3. Build Groups and Image URLs BEFORE clearing IDs
df['product_group'] = df['name'].notnull().cumsum()

df['image'] = 'https://media.githubusercontent.com/media/ahmedkhaled115/image-storage/main/' + \
              df['id'].astype(str).str.split('.').str[0] + '.jpg'

# 4. Update Categories (Mapping based on Keywords)
def map_category(name):
    if pd.isna(name): return ""
    name = str(name).lower()
    
    if "radiator" in name: return "Radiators"
    if "downpipe" in name or "decat" in name or "fooler" in name or "spacer" in name: return "Downpipes"
    if "intercooler" in name: return "Intercoolers"
    if "induction kit" in name or "air kit" in name: return "Air Kits"
    if "spoiler" in name or "wing" in name: return "Spoilers And Wings"
    if "grille" in name: return "Grilles"
    if "spring" in name: return "Lowering Springs"
    if "arm" in name or "rod" in name or "camber" in name: return "Adjustable arms"
    if "bilstein" in name: return "Bilstein Suspension Coilovers"
    if "brace" in name: return "Strut Braces"
    if "diffuser" in name or "canard" in name or "spat" in name: return "Rear Diffusers"
    if "intake" in name or "induction" in name or "filter" in name: return "Air Induction"
    if "exhaust" in name or "catback" in name or "back box" in name: return "Exhaust"
    return "Exhaust" 

df.loc[df['name'].notnull(), 'categ_id'] = df.loc[df['name'].notnull(), 'name'].apply(map_category)
df['public_categ_id'] = df.groupby('product_group')['categ_id'].transform('first')

# 5. Final Processing (Renaming, Cleaning, and Adding Published Column)

# Add Published Status
df['is_published'] = True

# Rename gallery media names
df['Extra Product Media/Name'] = df['Extra Product Media/Name'].str.replace('Gallery Image ', 'Name ', regex=False)

# image_1920 is the first image of the group
df['image_1920'] = df.groupby('product_group')['image'].transform('first')

# Identify rows that are NOT the first row of a product
is_gallery_row = df.duplicated(subset=['product_group'])

# Clear metadata for gallery rows (Added 'is_published' to this list)
cols_to_clear_gallery = [
    'name', 'public_categ_id', 'default_code', 'barcode', 'list_price', 
    'standard_price', 'weight', 'description', 'image_1920', 'is_published'
]

df.loc[is_gallery_row, cols_to_clear_gallery] = ""
df['id'] = "" 

# Reorder columns to place is_published at the end
column_order = [
    'id', 'name', 'public_categ_id', 'default_code', 'barcode', 'list_price', 
    'standard_price', 'weight', 'description', 'image_1920', 'image', 
    
    'Extra Product Media/Name', 'is_published'
]
df_final = df[column_order]

# 6. Split into Files (30 products per file)
products_per_file = 30
unique_product_ids = df['product_group'].unique()

for i in range(0, len(unique_product_ids), products_per_file):
    batch_ids = unique_product_ids[i : i + products_per_file]
    file_subset = df_final[df['product_group'].isin(batch_ids)]
    
    file_num = (i // products_per_file) + 1
    file_path = os.path.join(output_folder, f'Products_Batch_{file_num}.xlsx')
    
    file_subset.to_excel(file_path, index=False)
    print(f"File {file_num} saved to /data/ folder.")

print("\nDone! All files created with 'is_published' column set to TRUE.")

File 1 saved to /data/ folder.
File 2 saved to /data/ folder.
File 3 saved to /data/ folder.
File 4 saved to /data/ folder.
File 5 saved to /data/ folder.
File 6 saved to /data/ folder.
File 7 saved to /data/ folder.
File 8 saved to /data/ folder.
File 9 saved to /data/ folder.

Done! All files created with 'is_published' column set to TRUE.


/var/folders/vp/xy1fyb4j5f594ppbf387zs8c0000gn/T/ipykernel_81258/3302719551.py:63: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[is_gallery_row, cols_to_clear_gallery] = ""
/var/folders/vp/xy1fyb4j5f594ppbf387zs8c0000gn/T/ipykernel_81258/3302719551.py:63: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[is_gallery_row, cols_to_clear_gallery] = ""
/var/folders/vp/xy1fyb4j5f594ppbf387zs8c0000gn/T/ipykernel_81258/3302719551.py:63: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatibl